# 10-Worked Examples from Lectures

In [1]:
import sympy as sp
import sympy.physics.mechanics as spm
import sympy.physics.vector as spv
from sympy.physics.vector.printing import init_vprinting
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import IPython

# Import IPython display for proper LaTeX formatting
from IPython.display import display, Math, Markdown

# Initialize symbols
sp.init_printing()

# Enable dot notation printing for dynamicsymbols
init_vprinting(use_latex="mathjax")

HALF = sp.S.Half

In [2]:
def reference_frame(frame: str, x=r"\imath", y=r"\jmath", z=r"k") -> spm.ReferenceFrame:
    """Create a SymPy reference frame with custom basis vector labels.

    Parameters
    ----------
    frame : str
        The name of the reference frame.
    x, y, z : str
        Labels for the basis vectors.
    """
    return spm.ReferenceFrame(
        frame,
        latexs=(
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{x}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{y}}}",
            rf"\;{{}}^\mathcal{{{frame}}}\hat{{{z}}}",
        ),
    )


def reference_frame_circular(name: str, angle=r"theta") -> spm.ReferenceFrame:
    """Create a circular reference frame with radial and angular basis labels.

    Parameters
    ----------
    name : str
        Name of the new reference frame.
    angle : str, optional
        Symbol or label used for the angular basis vector, by default "theta".
    """
    return reference_frame(name, x=r"r", y=rf"\{angle}", z=r"e_z")

A uniform massive pulley of radius $R$ and moment of inertia $I_A$ about its center of mass  is suspended from a ceiling. An inextensible string of negligible mass is wrapped around the pulley and attached on one end to an object of mass  $m_1$ and on the other end to an object of mass $m_2$ .

Assume $m_2 \gt m_1$. Find the acceleration of the objects.

|Diagram|Coordinates|
|---|---|
|![Atwood machine](..//figures/WE3104-Atwood-Machine01.jpg)| ![Atwood machine](..//figures/WE3104-Atwood-Machine02.jpg)|

* Friction on Pulley
* Mass of pulley $m_A \gt 0$


In [3]:
(R, # Radius of the disc
 m1, # Mass of block 1
 m2, # Mass of block 2
 mA, # Mass of the pulley
 IA, # Moment of inertia of disc about the center of mass
 T1, # Tension in the string on block 1
 T2, # Tension in the string on block 2
 g # Acceleration due to gravity
) = sp.symbols('R m1 m2 m_A I_A T1 T2 g', real=True, positive=True)

N = reference_frame('N')
A = reference_frame("A")
theta = spv.dynamicsymbols("theta", real=True, positive=True)
A.orient_axis(N, N.z, theta)

R_vec = R * A.x
R_vel = R_vec.dt(N)
R_acc = R_vel.dt(N)


In [4]:
display(
    Markdown( 
        r"FBD for masses $m_1$ and $m_2$:"
        "\n\n"
        r"![FBD for masses](../figures/WE3104-Atwood-Machine05.jpg)"
    )
)

a = sp.symbols("a", real=True)
vec_forces_1 = T1*N.x - m1*g*N.x
vec_forces_2 = T2*N.x - m2*g*N.x

N2Leq1 = sp.Eq(vec_forces_1.dot(N.x), m1*a*N.x.dot(N.x))
N2Leq2 = sp.Eq(vec_forces_2.dot(N.x), m2*a*(-N.x).dot(N.x))

display(Math(r"\text{N2L on } m1: " rf"{spv.vlatex(N2Leq1)}"))
display(Math(r"\text{N2L on } m2: " rf"{spv.vlatex(N2Leq2)}"))

FBD for masses $m_1$ and $m_2$:

![FBD for masses](../figures/WE3104-Atwood-Machine05.jpg)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [5]:
display(
    Markdown( 
        r"Pulley kinetics :"
        "\n\n"
        r"![FBD for masses](../figures/WE3104-Atwood-Machine06.jpg)"
    )
)


Pulley kinetics :

![FBD for masses](../figures/WE3104-Atwood-Machine06.jpg)

In [6]:
vec_rT1 = R*(-N.y)
vec_rT2 = R*(N.y)

alpha = sp.symbols("alpha", real=True)
tau = vec_rT1.cross(T1*(-N.x)) + vec_rT2.cross(T2*(-N.x))
N2LReqn = sp.Eq(tau.dot(N.z), IA*alpha*N.z.dot(N.z))
display(Math(r"\text{N2L on pulley: }" rf"{spv.vlatex(N2LReqn)}"))

N2LReqn_alpha = N2LReqn.subs(alpha, a/R)
display(
    Math(
        r"\text{N2L on pulley (with } \alpha = a/R\text{): }" 
        rf"{spv.vlatex(N2LReqn_alpha)}"
    )
)

solutions = sp.solve([N2LReqn_alpha, N2Leq1, N2Leq2], [a, T1, T2])
solutions_simplified = { var: sol.simplify() for var, sol in solutions.items() }

display(Math(
    r"\text{Solutions: }" +"\\\\"+ " \\\\ ".join(
        rf"{spv.vlatex(var)} = {spv.vlatex(sol)}" for var, sol in solutions_simplified.items()
    )
))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [7]:
display(
    Markdown(
        "| Diagram | Coordinates |\n"
        "| :---: | :---: |\n"
        "| ![Atwood machine](../figures/WE3104-Atwood-Machine07.jpg) | ![Atwood machine](../figures/WE3104-Atwood-Machine02.jpg) |"
    )
)
t = spv.dynamicsymbols._t
ell = sp.symbols("ell", real=True, positive=True)
x2 = spv.dynamicsymbols("x2")
x1 = ell - x2

V1 = -m1 * g * x1
V2 = -m2 * g * x2
V = (V1 + V2).expand().collect(x2)
T = (
    HALF * m1 * x2.diff(t) ** 2
    + HALF * m2 * x2.diff(t) ** 2
    + HALF * IA * A.ang_vel_in(N).magnitude() ** 2
)
L = (T - V).subs(theta.diff(t), x2.diff(t)/R)

display(Math(r"\text{T: }" + rf"{spv.vlatex(T)}"))
display(Math(r"\text{V: }" + rf"{spv.vlatex(V)}"))
display(Math(r"\text{Lagrangian: }" + rf"{spv.vlatex(L)}"))



| Diagram | Coordinates |
| :---: | :---: |
| ![Atwood machine](../figures/WE3104-Atwood-Machine07.jpg) | ![Atwood machine](../figures/WE3104-Atwood-Machine02.jpg) |

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [8]:
display(
    Markdown(
        "The core formula for Lagrange's equation (commonly referred to as "
        "the Euler-Lagrange equation) for a system described by a set of "
        "generalized coordinates $q_i$ is:"
        "\n"
        r"$$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}_i}\right) - \frac{\partial L}{\partial q_i} = 0$$"
    )
)

Partial_L_x2dot = sp.diff(L, x2.diff(t))
Partial_L_x2 = sp.diff(L, x2)
Lagrange_eq_x2 = sp.Eq(sp.diff(Partial_L_x2dot, t) - Partial_L_x2, 0)
display(Math(r"\text{Lagrange equation for } x_2: " + rf"{spv.vlatex(Lagrange_eq_x2)}"))

acceleration = sp.solve(Lagrange_eq_x2, x2.diff(t, t))[0]
display(Math(r"\text{Acceleration of } x_2: " + rf"\boxed{{{spv.vlatex(acceleration)}}}"))

The core formula for Lagrange's equation (commonly referred to as the Euler-Lagrange equation) for a system described by a set of generalized coordinates $q_i$ is:
$$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}_i}\right) - \frac{\partial L}{\partial q_i} = 0$$

<IPython.core.display.Math object>

<IPython.core.display.Math object>